# 🧠 EX48: ค่าความแม่นยำเฉลี่ยเฉลี่ย (Mean Average Precision - mAP)
### *การบรรยายโดยศาสตราจารย์ด้านคอมพิวเตอร์วิทัศน์ของคุณ*

ยินดีต้อนรับกลับเข้าสู่บทเรียน! วันนี้เราจะมาศึกษาตัวชี้วัดที่เป็นเกณฑ์มาตรฐานสูงสุดสำหรับการตรวจจับวัตถุ: **ค่าความแม่นยำเฉลี่ยเฉลี่ย (Mean Average Precision - mAP)**.

ในการตรวจจับวัตถุ การประเมินแบบจำลองมีความซับซ้อนมากกว่าการจำแนกประเภททั่วไปอย่างมาก เนื่องจากเราต้องประเมินสองสิ่งนี้พร้อมกัน:
1. **ความถูกต้องของการจำแนกประเภท (Classification accuracy):** เราจำแนกวัตถุได้ถูกต้องคลาสหรือไม่?
2. **ความแม่นยำของการระบุตำแหน่ง (Localization precision):** เราวาดกล่องขอบเขตได้แม่นยำเพียงใด?

เรามาจำแนกตัวชี้วัดนี้ตั้งแต่คณิตศาสตร์พื้นฐานแรกเริ่มไปจนถึงการแสดงผลด้วยภาพกันเลย

---

## 📖 1. ตัวชี้วัดหลักจากหลักการพื้นฐาน

มาทบทวนคำจำกัดความมาตรฐานของ Precision และ Recall กัน:

- **ความแม่นยำ (Precision - P):** จากกล่องทำนายทั้งหมดที่แบบจำลองสร้างขึ้น มีเศษส่วนเท่าใดที่ถูกต้องจริง ๆ?
  $$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$
- **การระลึก (Recall - R):** จากวัตถุจริงทั้งหมดที่มีอยู่ในภาพ แบบจำลองค้นพบได้เป็นเศษส่วนเท่าใด?
  $$\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}} = \frac{\text{TP}}{\text{Total Ground Truths}}$$

### อะไรคือ "ผลบวกจริง" (True Positive - TP) ในการตรวจจับวัตถุ?
กล่องขอบเขตที่ทำนายได้ $B_{\text{pred}}$ จะเป็น **ผลบวกจริง (TP)** ก็ต่อเมื่อตรงตามเงื่อนไขต่อไปนี้ทั้งหมด:
1. ป้ายกำกับคลาสที่ทำนายตรงกับป้ายกำกับคลาสจริง (ground-truth class label)
2. พื้นที่ทับซ้อนเชิงพื้นที่ ซึ่งวัดโดยค่า **จุดตัดส่วนด้วยจุดรวม (Intersection over Union - IoU)** กับกล่องจริง $B_{\text{gt}}$ มีค่ามากกว่าหรือเท่ากับค่าเกณฑ์ $\tau$ ที่กำหนด (โดยทั่วไปคือ $\tau = 0.5$):
   $$\text{IoU}(B_{\text{pred}}, B_{\text{gt}}) = \frac{\text{Area}(B_{\text{pred}} \cap B_{\text{gt}})}{\text{Area}(B_{\text{pred}} \cup B_{\text{gt}}) } \ge \tau$$
3. เป็นการทำนายที่**มีคะแนนความมั่นใจสูงสุด**ที่จับคู่กับกล่องจริงนั้น หากมีกล่องทำนายหลายกล่องจับคู่กับกล่องจริงเดียวกัน จะมีเพียงกล่องที่มีความมั่นใจสูงสุดเท่านั้นที่จะถูกระบุว่าเป็น **TP** ส่วนกล่องทำนายอื่น ๆ ทั้งหมดที่จับคู่ซ้ำจะถูกระบุว่าเป็น **ผลบวกลวง (False Positives - FP)** รายละเอียดนี้สำคัญมากเพื่อป้องกันไม่ให้แบบจำลองสุ่มสร้างกล่องจำนวนมากเพื่อเพิ่มค่า Recall

หากกล่องทำนายไม่ผ่านเงื่อนไขใดเงื่อนไขหนึ่งข้างต้น จะถือว่าเป็น **ผลบวกลวง (FP)**
กล่องจริงที่ไม่ถูกจับคู่โดยกล่องทำนายใดเลยจะถือว่าเป็น **ผลลบลวง (FN)**

---

## 📐 2. พื้นที่ใต้เส้นโค้งและหลักคณิตศาสตร์ของการหาค่าเฉลี่ยแบบช่วง (Interpolation)

เพื่อคำนวณหาค่าความแม่นยำเฉลี่ย (Average Precision - AP) สำหรับแต่ละคลาส:
1. เราเรียงลำดับกล่องทำนายทั้งหมดในคลาสนั้นตามคะแนนความมั่นใจจากมากไปน้อย
2. We calculate cumulative TP and FP down the sorted list.
3. We compute the Precision and Recall values at each prediction step.
4. We construct a Precision-Recall (PR) curve.

เนื่องจากเส้นโค้ง PR ดิบมักจะมีลักษณะฟันปลาขึ้น ๆ ลง ๆ เราจึงทำให้มันเรียบขึ้นโดยใช้ **การหาค่า Precision แบบช่วงที่ราบเรียบ (Interpolated Precision)**:
$$P_{\text{interp}}(R) = \max_{\tilde{R} \ge R} P(\tilde{R})$$
ซึ่งหมายความว่าสำหรับระดับการระลึก $R$ ใด ๆ เราจะเลือกค่า Precision สูงสุดที่พบบนช่วง Recall ที่มีค่ามากกว่าหรือเท่ากับ $R$

### วิธี A: การหาค่าเฉลี่ยแบบช่วง 11 จุด (11-Point Interpolation - PASCAL VOC 2007)
เราเฉลี่ยค่า Interpolated Precision ที่ระดับการระลึก 11 จุดที่ห่างเท่า ๆ กัน: $0.0, 0.1, 0.2, \dots, 1.0$.
$$\text{AP}_{11} = \frac{1}{11} \sum_{r \in \{0.0, 0.1, \dots, 1.0\}} P_{\text{interp}}(r)$$

### วิธี B: การหาค่าเฉลี่ยแบบช่วงทุกจุด (All-Point Interpolation - COCO และ PASCAL VOC 2012)
แทนที่จะสุ่มเพียง 11 จุด เราจะทำการอินทิเกรตพื้นที่ทั้งหมดใต้เส้นโค้งที่เรียบขึ้น เราจะหาจุด Recall ที่ไม่ซ้ำกันทั้งหมดที่ทำให้ Precision ตก และแบ่งพื้นที่ออกเป็นรูปสี่เหลี่ยมผืนผ้า:
$$\text{AP} = \sum_{i} (R_{i+1} - R_i) P_{\text{interp}}(R_{i+1})$$

### จาก AP สู่ mAP
ค่าความแม่นยำเฉลี่ยเฉลี่ย (Mean Average Precision - mAP) คือค่าเฉลี่ยของ AP ของทุก ๆ $C$ คลาสในชุดข้อมูล:
$$\text{mAP} = \frac{1}{C} \sum_{c=1}^{C} \text{AP}_c$$

---

## 🔄 3. รูปแบบของค่า mAP ใน YOLO: mAP@0.5 เทียบกับ mAP@0.5:0.95

ในบันทึกผลการฝึกฝนของ YOLO ยุคใหม่ คุณจะเห็นตัวชี้วัด mAP หลักสองตัว:
- **mAP@0.5 (mAP50):** ค่า mAP ที่คำนวณที่เกณฑ์ IoU ค่าเดียวเท่ากับ 0.5 ตัวชี้วัดนี้ใช้วัดประสิทธิภาพของแบบจำลองในการตรวจพบและจำแนกประเภทวัตถุ (มีความผ่อนปรนค่อนข้างมากต่อการจัดวางตำแหน่งขอบเขต)
- **mAP@0.5:0.95 (mAP50-95):** ค่าเฉลี่ย mAP จากเกณฑ์ IoU 10 ค่าที่แตกต่างกัน ตั้งแต่ 0.50 ถึง 0.95 โดยมีระยะก้าวทีละ 0.05 (คือ 0.50, 0.55, 0.60, ..., 0.95) นี่คือตัวชี้วัดมาตรฐานของ COCO และมีความเข้มงวดสูงมาก โดยจะทำโทษแบบจำลองที่ทำนายตำแหน่งกล่องไม่ตรงกับตำแหน่งจริงอย่างแม่นยำ

เรามาเขียนโค้ดเพื่อนำหลักคณิตศาสตร์นี้มาใช้งานและแสดงภาพผลลัพธ์กัน

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def calculate_iou(box1, box2):
    """
    Calculates Intersection over Union (IoU) between box1 and box2.
    Format: [x1, y1, x2, y2]
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    union = area1 + area2 - intersection
    if union <= 0.0:
        return 0.0
    return intersection / union

def calculate_ap_single_class(gt_boxes, pred_boxes, iou_threshold=0.5, method="all-point"):
    """
    Calculates AP, cumulative precisions, and cumulative recalls for a single class.
    """
    num_gts = len(gt_boxes)
    num_preds = len(pred_boxes)
    
    if num_gts == 0:
        return 0.0, np.zeros(num_preds), np.zeros(num_preds)
    if num_preds == 0:
        return 0.0, np.array([]), np.array([])
        
    # Sort predictions by confidence score descending
    sorted_preds = sorted(pred_boxes, key=lambda x: x["confidence"], reverse=True)
    
    # Group ground truths by image_id
    gt_by_img = {}
    for gt in gt_boxes:
        img_id = gt["image_id"]
        if img_id not in gt_by_img:
            gt_by_img[img_id] = []
        gt_by_img[img_id].append(gt)
        
    # Track matched ground truths to prevent double-matching
    gt_matched = {}
    for img_id, boxes in gt_by_img.items():
        gt_matched[img_id] = [False] * len(boxes)
        
    tps = np.zeros(num_preds)
    fps = np.zeros(num_preds)
    
    # Determine TP/FP for each prediction
    for idx, pred in enumerate(sorted_preds):
        img_id = pred["image_id"]
        pred_box = pred["box"]
        
        if img_id not in gt_by_img or len(gt_by_img[img_id]) == 0:
            fps[idx] = 1.0
            continue
            
        best_iou = -1.0
        best_gt_idx = -1
        
        for gt_idx, gt in enumerate(gt_by_img[img_id]):
            iou = calculate_iou(pred_box, gt["box"])
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx
                
        if best_iou >= iou_threshold:
            if not gt_matched[img_id][best_gt_idx]:
                tps[idx] = 1.0
                gt_matched[img_id][best_gt_idx] = True
            else:
                fps[idx] = 1.0
        else:
            fps[idx] = 1.0
            
    cum_tps = np.cumsum(tps)
    cum_fps = np.cumsum(fps)
    
    precisions = cum_tps / (cum_tps + cum_fps)
    recalls = cum_tps / num_gts
    
    ap = 0.0
    if method == "11-point":
        for t in np.linspace(0.0, 1.0, 11):
            matching_indices = np.where(recalls >= t)[0]
            p_at_t = np.max(precisions[matching_indices]) if len(matching_indices) > 0 else 0.0
            ap += p_at_t / 11.0
    else:
        mrec = np.concatenate(([0.0], recalls, [1.0]))
        mpre = np.concatenate(([0.0], precisions, [0.0]))
        
        for i in range(len(mpre) - 2, -1, -1):
            mpre[i] = max(mpre[i], mpre[i+1])
            
        i = np.where(mrec[1:] != mrec[:-1])[0]
        ap = np.sum((mrec[i+1] - mrec[i]) * mpre[i+1])
        
    return ap, precisions, recalls

## 📊 4. การแสดงภาพ PR-Curve และการจับคู่กล่อง

มาประเมินชุดข้อมูลการทดสอบแบบคลาสเดี่ยว คำนวณค่า AP ของมัน และพลอตกราฟ:
1. **เส้นโค้ง Precision-Recall (แบบดิบ เทียบกับ แบบเฉลี่ยช่วงราบเรียบ)**
2. **การจับคู่กล่องขอบเขต (Bounding Box Matches)** เพื่อแสดงภาพพิกัดว่ากล่องทำนายจับคู่กับกล่องจริงอย่างไร

In [ ]:
# Verification data
gt_boxes = [
    {"image_id": 0, "box": [10, 10, 50, 50]},
    {"image_id": 0, "box": [60, 60, 100, 100]},
    {"image_id": 0, "box": [120, 120, 160, 160]}
]

pred_boxes = [
    {"image_id": 0, "box": [12, 12, 48, 48], "confidence": 0.95}, # TP
    {"image_id": 0, "box": [58, 58, 98, 98], "confidence": 0.88}, # TP
    {"image_id": 0, "box": [122, 122, 158, 158], "confidence": 0.75}, # TP
    {"image_id": 0, "box": [14, 14, 46, 46], "confidence": 0.65}, # FP (Duplicate)
    {"image_id": 0, "box": [200, 200, 240, 240], "confidence": 0.50}  # FP (No overlap)
]

ap, prec, rec = calculate_ap_single_class(gt_boxes, pred_boxes, iou_threshold=0.5, method="all-point")

# Create the dual-plot visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Precision-Recall Curve
# Interpolated curve coordinates for step plot
mrec = np.concatenate(([0.0], rec, [1.0]))
mpre = np.concatenate(([0.0], prec, [0.0]))
for i in range(len(mpre) - 2, -1, -1):
    mpre[i] = max(mpre[i], mpre[i+1])

axes[0].step(mrec, mpre, where='post', label='Interpolated PR Curve', color='#2ca02c', linewidth=3)
axes[0].plot(rec, prec, 'o--', label='Raw Predictions', color='#1f77b4', markersize=8, alpha=0.7)
axes[0].fill_between(mrec, mpre, step='post', alpha=0.15, color='#2ca02c')

axes[0].set_xlabel('Recall', fontsize=12)
axes[0].set_ylabel('Precision', fontsize=12)
axes[0].set_title(f'Precision-Recall Curve (AP = {ap:.4f})', fontsize=14, fontweight='bold')
axes[0].set_xlim(0, 1.05)
axes[0].set_ylim(0, 1.05)
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend(fontsize=11)

# Plot 2: Bounding Box Matches
axes[1].set_xlim(0, 250)
axes[1].set_ylim(250, 0) # Invert Y axis to mimic image coordinates
axes[1].set_title('Spatial Visualization of Bounding Boxes', fontsize=14, fontweight='bold')

# Draw Ground Truths (Green)
for idx, gt in enumerate(gt_boxes):
    box = gt["box"]
    rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                            linewidth=2, edgecolor='#2ca02c', facecolor='none', linestyle='-')
    axes[1].add_patch(rect)
    axes[1].text(box[0]+2, box[1]+15, f"GT {idx+1}", color='#2ca02c', fontweight='bold', fontsize=10)

# Draw Predictions (Blue/Red)
for idx, pred in enumerate(pred_boxes):
    box = pred["box"]
    conf = pred["confidence"]
    is_tp = idx in [0, 1, 2]
    edge_color = '#1f77b4' if is_tp else '#d62728'
    label = f"Pred {idx+1} ({conf:.2f}) [TP]" if is_tp else f"Pred {idx+1} ({conf:.2f}) [FP]"
    
    rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                            linewidth=2, edgecolor=edge_color, facecolor='none', linestyle='--')
    axes[1].add_patch(rect)
    axes[1].text(box[0]+2, box[3]-5, label, color=edge_color, fontweight='bold', fontsize=9)

axes[1].grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()